# Identify and Extract Job Ads in Historical Newspapers

## Get candidate pages from iiif manifest

In [1]:
import pandas as pd
import json
import requests
import PIL
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
import torch
from tqdm.notebook import trange, tqdm
import time
import shutil
import os
from io import BytesIO
import re

In [2]:
iiif_df = pd.read_csv("data_dh-infra.csv")

In [17]:
images_list = []
id_list = []
for index, row in tqdm(iiif_df.iterrows()):
    url = row['iiif_manifest']
    try:
        response = requests.get(url)
        data = response.json()
        for entry in data.get("metadata", []):
            label_de = entry.get("label", {}).get("de", [""])[0]
        
            if label_de == "Jahr/Datierung":
                edition_date = entry.get("value", {}).get("de", [""])[0]
                edition_year = pd.to_datetime(edition_date, dayfirst=True).strftime('%Y')
                edition_date = pd.to_datetime(edition_date, dayfirst=True).strftime('%d-%m-%Y')
                break
    
        images = [anno["body"]["id"] for canvas in data.get("items", []) for page in canvas.get("items", []) for anno in page.get("items", [])
                  if anno.get("motivation") == "painting"]
        images_id = [iiif_df['anno_id'][0]+f'_{i}' for i in range(0, len(images))]
        images_list.extend(images)
        id_list.extend(images_id)
    except requests.exceptions.RequestException as e:
        print(f"Error fetching the URL: {e}")
    except Exception as e:
        print(f"An error occurred: {e}")

0it [00:00, ?it/s]

In [18]:
page_df = pd.DataFrame({'anno_id': id_list, 'iiif_manifest': images_list})

In [19]:
page_df

,anno_id,iiif_manifest
0,guv18911215_0,https://api.onb.ac.at/iiif/image/v3/11F4D747/u...
1,guv18911215_1,https://api.onb.ac.at/iiif/image/v3/11F4D747/u...
2,guv18911215_2,https://api.onb.ac.at/iiif/image/v3/11F4D747/u...
3,guv18911215_3,https://api.onb.ac.at/iiif/image/v3/11F4D747/u...
4,guv18911215_4,https://api.onb.ac.at/iiif/image/v3/11F4D747/u...
5,guv18911215_5,https://api.onb.ac.at/iiif/image/v3/11F4D747/u...
6,guv18911215_6,https://api.onb.ac.at/iiif/image/v3/11F4D747/u...
7,guv18911215_7,https://api.onb.ac.at/iiif/image/v3/11F4D747/u...
8,guv18911215_8,https://api.onb.ac.at/iiif/image/v3/11F4D747/u...
9,guv18911215_9,https://api.onb.ac.at/iiif/image/v3/11F4D747/u...


## Check if candidate pages contain any job ads 

In [14]:
from transformers import AutoModelForImageClassification, AutoImageProcessor
from torch import nn

model_repo = 'Var3n/DiT_classification_jobads_finetuned'
image_processor = AutoImageProcessor.from_pretrained(model_repo)
model = AutoModelForImageClassification.from_pretrained(model_repo)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


BeitForImageClassification(
  (beit): BeitModel(
    (embeddings): BeitEmbeddings(
      (patch_embeddings): BeitPatchEmbeddings(
        (projection): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): BeitEncoder(
      (layer): ModuleList(
        (0): BeitLayer(
          (attention): BeitAttention(
            (attention): BeitSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=False)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (output): BeitSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): BeitIntermediate(
            (dense): Lin

In [15]:
os.makedirs('Backup', exist_ok=True)

In [ ]:
images = page_df['iiif_manifest'].to_list()

In [16]:
n = 25 
img_batches = [images[i:i+n] for i in range(0, len(images), n)]
id_batches = [images_id[i:i+n] for i in range(0, len(images_id), n)]
back_up = 0
for ind, batch in tqdm(enumerate(img_batches), total=len(img_batches)):
    id_batch = id_batches[ind]
    images_data = []
    preds = []
    probs = []
    restart = False 
    retry = 0
    for image in batch:
        while True:
            try:
                img_data = requests.get(image)
                img_data = PIL.Image.open(BytesIO(img_data.content)).convert('RGB')
                images_data.append(img_data)
                if restart:
                    t = time.localtime()
                    current_time = time.strftime('%d-%m %H:%M:%S', t)
                    restart = False
                    retry = 0
                    print(f'{current_time}: Reconnected')
            except OSError as e1:
                t = time.localtime()
                current_time = time.strftime('%d-%m %H:%M:%S', t)
                print(f'{current_time}: {e1} \n Rate limit. Retry after 1 minute.')
                time.sleep(60)
                restart = True
                if retry == 3:
                    print(f'Tried {retry} times. Skipping image.')
                    break
                retry += 1
                continue
            else:
                break
    encoding = image_processor(images_data, return_tensors = 'pt')
    encoding = encoding.to(device)

    with torch.no_grad():
        outputs = model(**encoding)
        logits = outputs.logits
    preds.extend(logits.argmax(-1).tolist())
    probs.extend(nn.functional.softmax(outputs.logits, dim = -1).amax(-1).tolist())
    with open('all_ads.txt', 'a') as f:
        for (image_id, image, pred, prob) in zip(id_batch, batch, preds, probs):
            f.write('{0} {1} {2} {3}\n'.format(image_id, image, pred, prob))
    del encoding
    torch.cuda.empty_cache()
    back_up += 1
    if back_up == 100:
        now = datetime.now()
        dest = 'Backup/all_ads_'+now.strftime('%d_%m_%Y_%H_%M')+'.txt'
        shutil.copy('all_ads.txt', dest)
        back_up = 0

  0%|          | 0/1 [00:00<?, ?it/s]

## Layout Analysis 

In [1]:
import os
import subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import re
from pathlib import Path
import threading
from collections import Counter

In [2]:
def pick_gpu():
    with gpu_lock:
        gpu_id = min(gpu_load, key=gpu_load.get)
        gpu_load[gpu_id] += 1
        return gpu_id

def release_gpu(gpu_id):
    with gpu_lock:
        gpu_load[gpu_id] -= 1

def run_year(label, file_list):
    gpu_id = pick_gpu()
    try:
        log_path = LOG_DIR / f"{label}.log"
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
        env["CONDA_PREFIX"] = str(EYNOLLAH_PREFIX)
        env["PATH"] = f"{EYNOLLAH_PREFIX}/bin:" + env.get("PATH", "")

        with open(log_path, "w") as logf:
            logf.write(f"[orchestrator] {label} pinned to GPU {gpu_id}\n")
            logf.flush()
            result = subprocess.run(["bash", str(SCRIPT), file_list], stdout=logf, stderr=subprocess.STDOUT, env=env)
        return label, gpu_id, result.returncode
    finally:
        release_gpu(gpu_id)

In [3]:
WORK_DIR = Path.cwd()
EYNOLLAH_PREFIX = Path("/home/lab6/miniforge3/envs/eyn_env")
SCRIPT = Path("layout_analysis.sh").resolve()          
LOG_DIR = WORK_DIR / "logs"
LOG_DIR.mkdir(exist_ok=True)

N_GPUS = 2
JOBS_PER_GPU = 1
MAX_PARALLEL = N_GPUS * JOBS_PER_GPU

In [4]:
gpu_load = Counter({i: 0 for i in range(N_GPUS)})
gpu_lock = threading.Lock()

In [5]:
pattern = re.compile(r"^(?P<short_title>[a-zA-Z]+)_(?P<year>\d{4})\.txt$")

year_to_filelist = {}
for path in sorted(WORK_DIR.glob("*.txt")):
    m = pattern.match(path.name)
    if not m:
        continue  # skip unrelated .txt files in the working dir
    label = f"{m['short_title']}_{m['year']}"
    year_to_filelist[label] = str(path)

print(f"Found {len(year_to_filelist)} file lists:")
for label in year_to_filelist:
    print(f"  {label}")

Found 44 file lists:
  ptb_1894
  ptb_1895
  ptb_1896
  ptb_1897
  ptb_1898
  ptb_1899
  ptb_1900
  ptb_1901
  ptb_1902
  ptb_1903
  ptb_1904
  ptb_1905
  ptb_1906
  ptb_1907
  ptb_1908
  ptb_1909
  ptb_1910
  ptb_1911
  ptb_1912
  ptb_1913
  ptb_1914
  ptb_1915
  ptb_1916
  ptb_1917
  ptb_1918
  ptb_1919
  ptb_1920
  ptb_1921
  ptb_1922
  ptb_1923
  ptb_1924
  ptb_1925
  ptb_1926
  ptb_1927
  ptb_1928
  ptb_1929
  ptb_1930
  ptb_1931
  ptb_1932
  ptb_1933
  ptb_1934
  ptb_1936
  ptb_1937
  ptb_1938


In [ ]:
jobs = list(year_to_filelist.items()) 

with ThreadPoolExecutor(max_workers=MAX_PARALLEL) as executor:
    futures = {executor.submit(run_year, label, fl): label for label, fl in year_to_filelist.items()}

    failed = []
    with tqdm(total=len(futures), desc="Years processed") as pbar:
        for future in as_completed(futures):
            label, gpu_id, rc = future.result()
            if rc != 0:
                failed.append(label)
                pbar.write(f"✗ {label} (GPU {gpu_id}) — see {LOG_DIR}/{label}.log")
            else:
                pbar.write(f"✓ {label} (GPU {gpu_id})")
            pbar.update(1)

if failed:
    print(f"\n{len(failed)} job(s) failed: {failed}")
else:
    print("\nAll jobs completed successfully.")

Years processed:   0%|          | 0/44 [00:00<?, ?it/s]

## OCR

In [1]:
import os
from pathlib import Path
from tqdm.auto import tqdm
from ocr_pipeline import run_pipeline, gather_processed_stems

In [2]:
PNG_BASE = "/home/lab6/nextcloud_fast/Jobads/final_selection"
JPG_BASE = "/home/lab6/nextcloud_fast/Jobads/preselected_pages"

def image_base_fn(rel_image_path: str) -> str:
    return JPG_BASE if rel_image_path.lower().endswith(".jpg") else PNG_BASE

In [3]:
rootdir     = "/home/lab6/nextcloud_fast/Jobads/final_selection"
paper_name  = "ptb"
input_root  = f"{rootdir}/{paper_name}"
output_root = f"{rootdir}/{paper_name}_ocr"

xml_files = []
with tqdm(desc="Scanning input", unit=" file") as pbar:
    for root, _, files in os.walk(input_root):
        for f in files:
            if f.endswith(".xml"):
                xml_files.append(os.path.join(root, f))
                pbar.update(1)

Scanning input: 0 file [00:00, ? file/s]

In [17]:
processed = gather_processed_stems(output_root)
todo = [p for p in xml_files if Path(p).stem not in processed]
print(f"{len(xml_files)} total | {len(processed)} done | {len(todo)} to process")

18795 total | 0 done | 18795 to process


In [18]:
errors = run_pipeline(todo, remote_output_root = output_root, image_base_fn = image_base_fn, tessdata_path = "/usr/share/tesseract-ocr/5/tessdata",
                      model_name = "frak2021-0.905", ocr_workers = 100, io_workers = 5, max_in_flight = 300)

Scratch: /dev/shm/ocr_dxp0ymul


OCR pipeline:   0%|          | 0/18795 [00:00<?, ? file/s]

All files processed successfully.


In [87]:
corrupted_xmls = [path for path, _ in errors]
corrupted_stems = {Path(p).stem for p in corrupted_xmls}
print(f"{len(corrupted_xmls)} corrupted XMLs to redo")

116 corrupted XMLs to redo


In [88]:
jpg_by_stem = {}
with open("rpt_files.txt") as f:
    for line in f:
        line = line.strip()
        if line:
            jpg_by_stem[Path(line).stem] = line

In [89]:
matched, unmatched = [], []
for stem in corrupted_stems:
    (matched if stem in jpg_by_stem else unmatched).append(stem)

redo_path = "rpt_files_redo.txt"
with open(redo_path, "w") as f:
    for stem in matched:
        f.write(jpg_by_stem[stem] + "\n")

print(f"Wrote {len(matched)} JPG paths to {redo_path}")
if unmatched:
    print(f"Warning: {len(unmatched)} stems had no match in rpt_files.txt, e.g.:")
    for s in unmatched[:5]:
        print(f"  {s}")

Wrote 116 JPG paths to rpt_files_redo.txt


In [90]:
deleted_xml = deleted_png = 0
errors_during_delete = []
for xml_path in corrupted_xmls:
    p = Path(xml_path)
    try:
        if p.exists():
            p.unlink()
            deleted_xml += 1
    except Exception as e:
        errors_during_delete.append((xml_path, str(e)))

    png = p.with_suffix(".png")
    try:
        if png.exists():
            png.unlink()
            deleted_png += 1
    except Exception as e:
        errors_during_delete.append((str(png), str(e)))

print(f"Deleted {deleted_xml} XML files and {deleted_png} PNG files")
if errors_during_delete:
    print(f"{len(errors_during_delete)} delete errors:")
    for p, e in errors_during_delete[:5]:
        print(f"  {p}: {e}")

Deleted 116 XML files and 0 PNG files


### Adjust file names of pngs enhanced by Eynollah

In [1]:
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import requests
from requests.auth import HTTPBasicAuth

In [2]:
def to_webdav_url(local_path):
    rel = local_path[len(LOCAL_PREFIX):].lstrip("/")
    # URL-encode path segments to handle spaces and special chars
    return WEBDAV_BASE + "/" + "/".join(requests.utils.quote(seg) for seg in rel.split("/"))

def rename_webdav(pair):
    src, dst = pair
    try:
        r = session.request(
            "MOVE",
            to_webdav_url(src),
            headers={"Destination": to_webdav_url(dst), "Overwrite": "F"},
            timeout=60,
        )
        if r.status_code in (201, 204):
            return True, None
        return False, f"HTTP {r.status_code}"
    except Exception as e:
        return False, str(e)

In [3]:
# Enter name of paper of newspaper to process
paper_name = 'ptb'

In [7]:
rootdir = '/home/lab6/nextcloud_fast/Jobads/final_selection'

to_rename = []
with tqdm(desc="Scanning", unit=" files") as pbar:
    for root, _, files in os.walk(f'{rootdir}/{paper_name}'):
        for file in files:
            if file.endswith('.png') and '_enhanced' in file:
                src = os.path.join(root, file)
                dst = src.replace('_enhanced', '')
                to_rename.append((src, dst))
                pbar.update(1)

Scanning: 0 files [00:00, ? files/s]

In [ ]:
import requests
from requests.auth import HTTPBasicAuth

WEBDAV_BASE = "" #adjust with your credentials and environment setup
USERNAME = ""
PASSWORD = ""
LOCAL_PREFIX = ""

session = requests.Session()
session.auth = HTTPBasicAuth(USERNAME, PASSWORD)
adapter = requests.adapters.HTTPAdapter(pool_connections=32, pool_maxsize=32)
session.mount("https://", adapter)

In [6]:
successful = 0
skipped = 0
errors = []
with ThreadPoolExecutor(max_workers=8) as ex:
    futures = [ex.submit(rename_webdav, pair) for pair in to_rename]
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Renaming"):
        ok, err = fut.result()
        if ok:
            successful += 1
        else:
            skipped += 1
            if err and "404" not in err:
                errors.append(err)

print(f"Renamed: {successful} | Skipped: {skipped}")
if errors:
    from collections import Counter
    print("Error breakdown:", Counter(errors).most_common(10))

Renaming:   0%|          | 0/18828 [00:00<?, ?it/s]

Renamed: 18826 | Skipped: 2
Error breakdown: [('HTTP 500', 2)]


### Change file paths inside xml files to enhanced png

In [9]:
import os
import time
import concurrent.futures
from pathlib import Path
from tqdm.auto import tqdm

from ocrd_models.ocrd_page import parse, to_xml

In [10]:
def process_single_file(xml_file):
    for attempt in range(MAX_RETRIES):
        try:
            pcgts = parse(xml_file, silence=True)
            page = pcgts.get_Page()

            old_filename = Path(page.get_imageFilename())

            if old_filename.suffix.lower() == '.png':
                return ("skipped", xml_file, "already PNG")

            if len(old_filename.parts) < 4:
                return ("error", xml_file, "path too short")

            new_filename = (
                f"{old_filename.parts[-4]}/"
                f"{old_filename.parts[-3]}/"
                f"{old_filename.stem}.png"
            )
            page.set_imageFilename(new_filename)

            tmp = xml_file + ".tmp"
            with open(tmp, "w", encoding="utf-8") as f:
                f.write(to_xml(pcgts))
            os.replace(tmp, xml_file)

            return ("success", xml_file, None)

        except TRANSIENT_ERRORS as e:
            if attempt < MAX_RETRIES - 1:
                time.sleep(RETRY_PAUSE)
                continue
            return ("error", xml_file, f"I/O failed after {MAX_RETRIES}: {e}")
        except Exception as e:
            return ("error", xml_file, f"{type(e).__name__}: {e}")

In [11]:
rootdir     = "/home/lab6/nextcloud_fast/Jobads/final_selection"
paper_name  = "ptb"
input_root  = f"{rootdir}/{paper_name}"

enhan_imgs = []
with tqdm(desc="Scanning input", unit=" file") as pbar:
    for root, _, files in os.walk(input_root):
        for f in files:
            if f.endswith(".png"):
                enhan_imgs.append(os.path.join(root, f))
                pbar.update(1)

xml_files = [img_file.replace(".png", ".xml") for img_file in enhan_imgs]

Scanning input: 0 file [00:00, ? file/s]

In [12]:
xml_files[:5]

['/home/lab6/nextcloud_fast/Jobads/final_selection/ptb/1877/ptb_18770111_007.xml',
 '/home/lab6/nextcloud_fast/Jobads/final_selection/ptb/1877/ptb_18770112_008.xml',
 '/home/lab6/nextcloud_fast/Jobads/final_selection/ptb/1877/ptb_18770116_007.xml',
 '/home/lab6/nextcloud_fast/Jobads/final_selection/ptb/1877/ptb_18770117_008.xml',
 '/home/lab6/nextcloud_fast/Jobads/final_selection/ptb/1877/ptb_18770118_007.xml']

In [13]:
MAX_RETRIES = 3
RETRY_PAUSE = 1.0
MAX_CONCURRENT_THREADS = 16
TRANSIENT_ERRORS = (OSError, IOError, TimeoutError)

print(f"Processing {len(xml_files)} files with {MAX_CONCURRENT_THREADS} threads...")

results = {"success": 0, "skipped": 0, "error": 0}
errors = []

with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_CONCURRENT_THREADS) as ex:
    futures = [ex.submit(process_single_file, f) for f in xml_files]
    for fut in tqdm(concurrent.futures.as_completed(futures),
                    total=len(futures), desc="Updating XMLs"):
        status, path, msg = fut.result()
        results[status] += 1
        if status == "error":
            errors.append((path, msg))

print("-" * 30)
print(f"Success: {results['success']} | Skipped: {results['skipped']} | Errors: {results['error']}")
if errors:
    print(f"\nFirst 10 errors:")
    for path, msg in errors[:10]:
        print(f"  {path}: {msg}")

Processing 18828 files with 16 threads...


Updating XMLs:   0%|          | 0/18828 [00:00<?, ?it/s]

------------------------------
Success: 18793 | Skipped: 0 | Errors: 35

First 10 errors:
  /home/lab6/nextcloud_fast/Jobads/final_selection/ptb/1878/ptb_18780316_010.xml: I/O failed after 3: Error reading file '/home/lab6/nextcloud_fast/Jobads/final_selection/ptb/1878/ptb_18780316_010.xml': failed to load "/home/lab6/nextcloud_fast/Jobads/final_selection/ptb/1878/ptb_18780316_010.xml": No such file or directory
  /home/lab6/nextcloud_fast/Jobads/final_selection/ptb/1878/ptb_18781202_006.xml: I/O failed after 3: Error reading file '/home/lab6/nextcloud_fast/Jobads/final_selection/ptb/1878/ptb_18781202_006.xml': failed to load "/home/lab6/nextcloud_fast/Jobads/final_selection/ptb/1878/ptb_18781202_006.xml": No such file or directory
  /home/lab6/nextcloud_fast/Jobads/final_selection/ptb/1880/ptb_18800103_011.xml: I/O failed after 3: Error reading file '/home/lab6/nextcloud_fast/Jobads/final_selection/ptb/1880/ptb_18800103_011.xml': failed to load "/home/lab6/nextcloud_fast/Jobads/final_

In [60]:
PAPER_NAME = "rpt"
ORIGINAL_LIST = "rpt_files.txt"
REGENERATE_LIST = "rpt_to_regenerate.txt"
 
OLD_PATH_PREFIX = "/home/lab6/nextcloud_fast/"
NEW_PATH_PREFIX = "/home/lab6/nextcloud_fast/"

In [61]:
CORRUPT_KEYWORDS = ("XMLSyntaxError", "Extra content", "not well-formed", "parse")
MISSING_KEYWORDS = ("No such file or directory", "FileNotFoundError")
 
 
def classify_errors(errors):
    corrupted, missing, other = [], [], []
    for path, msg in errors:
        if any(k in msg for k in MISSING_KEYWORDS):
            missing.append(path)
        elif any(k in msg for k in CORRUPT_KEYWORDS):
            corrupted.append(path)
        else:
            other.append((path, msg))
    return corrupted, missing, other

In [62]:
corrupted_xmls, missing_xmls, other_errors = classify_errors(errors)
print(f"Corrupted XMLs (will delete xml + png): {len(corrupted_xmls)}")
print(f"Missing XMLs   (will delete png only):  {len(missing_xmls)}")
print(f"Other errors   (will be ignored):       {len(other_errors)}")
if other_errors:
    print("\nFirst 5 'other' errors — review and add keywords if any should be handled:")
    for p, m in other_errors[:5]:
        print(f"  {p}: {m[:120]}")

broken_xmls = list(dict.fromkeys(corrupted_xmls + missing_xmls))

Corrupted XMLs (will delete xml + png): 11
Missing XMLs   (will delete png only):  7
Other errors   (will be ignored):       0


In [63]:
with open(ORIGINAL_LIST, "r") as f:
    original_jpgs = [line.strip() for line in f if line.strip()]
print(f"\nLoaded {len(original_jpgs)} original jpg paths from {ORIGINAL_LIST}")
 
stem_to_jpg = {}
duplicates = 0
for jpg in original_jpgs:
    stem = Path(jpg).stem
    if stem in stem_to_jpg:
        duplicates += 1
    stem_to_jpg[stem] = jpg
if duplicates:
    print(f"Warning: {duplicates} duplicate stems in {ORIGINAL_LIST}")


Loaded 34537 original jpg paths from rpt_files.txt


In [64]:
to_regenerate = []
matched_corrupted = []
matched_missing = []
unmatched = []
 
for xml_path in broken_xmls:
    stem = Path(xml_path).stem
    original_jpg = stem_to_jpg.get(stem)
    if original_jpg is None:
        unmatched.append(xml_path)
        continue
    fast_jpg = original_jpg.replace(OLD_PATH_PREFIX, NEW_PATH_PREFIX, 1)
    to_regenerate.append(fast_jpg)
    if xml_path in corrupted_xmls:
        matched_corrupted.append(xml_path)
    else:
        matched_missing.append(xml_path)
 
print(f"\nMatched {len(to_regenerate)} broken XMLs to original jpgs")
print(f"  corrupted (xml + png deletion): {len(matched_corrupted)}")
print(f"  missing   (png-only deletion):  {len(matched_missing)}")
if unmatched:
    print(f"\nCould not match {len(unmatched)} XMLs — they will NOT be deleted:")
    for p in unmatched[:10]:
        print(f"  {p}")
 
with open(REGENERATE_LIST, "w") as f:
    for path in to_regenerate:
        f.write(path + "\n")
print(f"\nWrote {len(to_regenerate)} paths to {REGENERATE_LIST}")


Matched 16 broken XMLs to original jpgs
  corrupted (xml + png deletion): 11
  missing   (png-only deletion):  5

Could not match 2 XMLs — they will NOT be deleted:
  /home/lab6/nextcloud_fast/Jobads/final_selection/rpt/1929/rpt_19290317_029_enhanced.xml
  /home/lab6/nextcloud_fast/Jobads/final_selection/rpt/1931/rpt_19310210_011_enhanced.xml

Wrote 16 paths to rpt_to_regenerate.txt


In [58]:
def derive_png_path(xml_path):
    """The PNG sits next to the XML with the same stem."""
    return str(Path(xml_path).with_suffix(".png"))

In [59]:
deleted_xml = 0
deleted_png = 0
missing_png = 0
errors_del = []

# Corrupted: delete both XML and PNG
for xml_path in tqdm(matched_corrupted, desc="Deleting corrupted"):
    try:
        os.remove(xml_path)
        deleted_xml += 1
    except FileNotFoundError:
        pass  # already gone, fine
    except Exception as e:
        errors_del.append((xml_path, str(e)))
        continue

    png_path = derive_png_path(xml_path)
    try:
        os.remove(png_path)
        deleted_png += 1
    except FileNotFoundError:
        missing_png += 1
    except Exception as e:
        errors_del.append((png_path, str(e)))

# Missing XMLs: delete just the orphaned PNG
for xml_path in tqdm(matched_missing, desc="Deleting orphan PNGs"):
    png_path = derive_png_path(xml_path)
    try:
        os.remove(png_path)
        deleted_png += 1
    except FileNotFoundError:
        missing_png += 1
    except Exception as e:
        errors_del.append((png_path, str(e)))

print(f"\nDeleted {deleted_xml} XMLs, {deleted_png} PNGs "
      f"({missing_png} PNGs were already missing)")
if errors_del:
    print(f"{len(errors_del)} deletion errors:")
    for p, e in errors_del[:10]:
        print(f"  {p}: {e}")

Deleting corrupted:   0%|          | 0/11 [00:00<?, ?it/s]

Deleting orphan PNGs:   0%|          | 0/5 [00:00<?, ?it/s]


Deleted 11 XMLs, 16 PNGs (0 PNGs were already missing)


## Identify Job Ad regions

In [1]:
from ocrd_models.ocrd_page import parse
import pandas as pd
from concurrent.futures import ProcessPoolExecutor
from tqdm.auto import tqdm
import os
from pathlib import Path
from gpu_classify import classify_dataframe

ModuleNotFoundError: No module named 'ocrd_models'

In [191]:
def _region_ids_in_reading_order(page, regions):
    ro = page.get_ReadingOrder()
    if ro is None:
        return list(regions.keys())
    root = ro.get_OrderedGroup() or ro.get_UnorderedGroup()
    if root is None:
        return list(regions.keys())
    ordered = []
    _walk_group(root, ordered)
    mentioned = set(ordered)
    for rid in regions:
        if rid not in mentioned:
            ordered.append(rid)
    return ordered


def _walk_group(group, out):
    def safe(name):
        return getattr(group, name, lambda: None)() or []
    items = []
    for ref in safe("get_RegionRefIndexed"):
        items.append((ref.index, "ref", ref.regionRef))
    for ref in safe("get_RegionRef"):
        items.append((None, "ref", ref.regionRef))
    for sub in safe("get_OrderedGroupIndexed"):
        items.append((sub.index, "group", sub))
    for sub in safe("get_UnorderedGroupIndexed"):
        items.append((sub.index, "group", sub))
    for sub in safe("get_OrderedGroup"):
        items.append((None, "group", sub))
    for sub in safe("get_UnorderedGroup"):
        items.append((None, "group", sub))
    if any(idx is not None for idx, _, _ in items):
        items.sort(key=lambda x: (x[0] is None, x[0]))
    for _, kind, payload in items:
        if kind == "ref":
            out.append(payload)
        else:
            _walk_group(payload, out)

def _coords_bbox(region):
    """min_x, max_x, min_y, max_y from a region's Coords, or None."""
    coords = region.get_Coords()
    if coords is None or not coords.points:
        return None
    pts = [tuple(map(int, p.split(","))) for p in coords.points.split()]
    xs, ys = [p[0] for p in pts], [p[1] for p in pts]
    return min(xs), max(xs), min(ys), max(ys)

def extract_regions_in_reading_order(xml_path):
    pcgts = parse(xml_path, silence=True)
    page = pcgts.get_Page()
    regions = {r.id: r for r in page.get_AllRegions(classes=["Text"])}
    ordered_ids = _region_ids_in_reading_order(page, regions)   # reuse your helper
    result = []
    for order_idx, rid in enumerate(ordered_ids):
        region = regions.get(rid)
        if region is None:
            continue
        lines = []
        for line in region.get_TextLine():
            equivs = line.get_TextEquiv()
            if equivs and equivs[0].Unicode:
                lines.append(equivs[0].Unicode)
        if not lines:
            continue
        bbox = _coords_bbox(region)
        if bbox is None:
            continue
        min_x, max_x, min_y, max_y = bbox
        result.append({
            "region_id": rid,
            "reading_order": order_idx,
            "text": "\n".join(lines),
            "min_x": min_x, "max_x": max_x,
            "min_y": min_y, "max_y": max_y,
        })
    return result

In [192]:
def _extract_rows(xml_path):
    try:
        regions = extract_regions_in_reading_order(xml_path)
    except Exception as e:
        return [{"xml_path": xml_path, "region_id": None, "reading_order": None,
                 "text": None, "min_x": None, "max_x": None,
                 "min_y": None, "max_y": None, "error": repr(e)}]
    return [{"xml_path": xml_path, **r, "error": None} for r in regions]

def build_region_dataframe(xml_files, workers=32):
    all_rows = []
    with ProcessPoolExecutor(max_workers=workers) as pool:
        for rows in tqdm(pool.map(_extract_rows, xml_files),
                         total=len(xml_files), desc="Extracting regions"):
            all_rows.extend(rows)
    return pd.DataFrame(all_rows)

In [287]:
tag = 'wrz'

True

In [281]:
input_root  = f"/home/lab6/nextcloud_fast/Jobads/final_selection/{tag}_ocr"  # relative to cwd, or use an absolute path

xml_files = []
with tqdm(desc="Scanning XMLs", unit=" file") as pbar:
    for root, _, files in os.walk(input_root):
        for f in files:
            if f.endswith(".xml"):
                xml_files.append(os.path.join(root, f))
                pbar.update(1)
print(f"Found {len(xml_files)} XML files")

Scanning XMLs: 0 file [00:00, ? file/s]

Found 24874 XML files


In [282]:
df = build_region_dataframe(xml_files, workers=32)

Extracting regions:   0%|          | 0/24874 [00:00<?, ?it/s]

In [283]:
classified = classify_dataframe(df, model_dir="/home/lab6/nextcloud_fast/Jobads/JobAdsIdentification/model_best/results_zeitungs/best", 
                                tokenizer_dir="stefan-it/zeitungs-lm-v1", batch_size=384)

Classifying (2 GPUs):   0%|          | 0/878502 [00:00<?, ?it/s]

In [284]:
job_ads = classified[classified["prediction"] == 1]
results = [{"metadata": Path(row["xml_path"]).stem, "region_id": row["region_id"], "reading_order": row["reading_order"], "text": row["text"], 
            "min_x": row["min_x"], "max_x": row["max_x"], "min_y": row["min_y"], "max_y": row["max_y"]} for _, row in job_ads.iterrows()]

In [285]:
ads_df = pd.DataFrame(results)

In [286]:
ads_df.to_csv(f"/home/lab6/nextcloud_fast/Jobads/final_selection/finished_csv/{tag}_classified.csv", index=False)

## Post-Correction

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import numpy as np
from tqdm.auto import tqdm
#from infer import parallel_generate_texts
import os
import pandas as pd

In [2]:
def byt5_byte_len(text):
    if not text:
        return 1
    return len(text.encode("utf-8")) + 1


def split_for_byt5(text, max_len=150, _seps=("\n\n", "\n", ". ", "; ", ", ", " ")):
    """
    Recursively split text so each chunk's ByT5 input length is <= max_len.
    Tries higher-quality separators first (paragraph -> sentence -> clause -> word),
    falls back to a UTF-8-safe byte-level split only for unbreakable runs.
    """
    if not text:
        return [text]
    if byt5_byte_len(text) <= max_len:
        return [text]

    # Try each separator in priority order; pick the first that yields a real split
    for sep in _seps:
        if sep not in text:
            continue
        parts = text.split(sep)
        if len(parts) < 2:
            continue
        mid = len(parts) // 2
        left = sep.join(parts[:mid])
        right = sep.join(parts[mid:])
        if left and right:
            return (split_for_byt5(left,  max_len, _seps)
                  + split_for_byt5(right, max_len, _seps))

    # No usable separator (single very long token): split at a UTF-8 char boundary
    encoded = text.encode("utf-8")
    mid = len(encoded) // 2
    # Walk forward past UTF-8 continuation bytes (0b10xxxxxx)
    while mid < len(encoded) and (encoded[mid] & 0xC0) == 0x80:
        mid += 1
    left  = encoded[:mid].decode("utf-8", errors="ignore")
    right = encoded[mid:].decode("utf-8", errors="ignore")
    return split_for_byt5(left, max_len, _seps) + split_for_byt5(right, max_len, _seps)

In [4]:
tag = 'nwb'
os.path.exists(f"{tag}_postcorrected.csv")

True

In [5]:
df = pd.read_csv(f"{tag}_classified.csv")

In [6]:
df

,metadata,region_id,reading_order,text,min_x,max_x,min_y,max_y
0,nwb_18740110_006,region_0024,28,"Josefstadt, Langegasse 55,\nempfiehlt ſich ſow...",806,1258,4231,4380
1,nwb_18740120_008,region_0065,18,"gutem Hauſe, wenn auch\nWaiſe und alleinſtehen...",802,1264,944,1374
2,nwb_18740120_014,region_0099,102,"Jouefstadt, Lansegasse 55.\nempflehlt ſich ſew...",2743,3200,4528,4678
3,nwb_18740121_007,region_0087,86,Das bürgl.\nDienstvermittlungs- Institut\nWien...,2618,3067,4402,4655
4,nwb_18740121_008,region_0059,17,"gutem Hauſe, wenn auch\nWaiſe und alleinſtehen...",759,1216,893,1321
...,...,...,...,...,...,...,...,...
175451,nwb_19431231_014,region_0082,206,"ſowie Serviererin,\nlagsüber, geſuchr.\nCafé G...",4447,4814,3197,3391
175452,nwb_19431231_014,region_0112,207,"Bedienerin\noder BVürodiener\nfür, ganz⸗ oder\...",4444,4818,3425,3743
175453,nwb_19431231_014,region_0178,208,Chrliche\nWirlſchaftlerin mit\nMochtanntniſſen...,4443,4820,3799,4148
175454,nwb_19431231_014,region_0185,210,Wienen Bühne ſüchi\nſlüüir Wehrmachts⸗\n(Hurnd...,4444,4816,4242,4579


In [5]:
df['text'] = df['text'].str.replace(r'ﬁ', r'fi', regex=True)
df['text'] = df['text'].str.replace(r'ﬂ', r'fl', regex=True)
df['text'] = df['text'].str.replace(r'¬', r'-', regex=True)
df['text'] = df['text'].str.replace(r'¶', r'', regex=True)
df['text'] = df['text'].str.replace(r'˖', r'+', regex=True)
df['text'] = df['text'].str.replace(r'‟', r'"', regex=True)
df['text'] = df['text'].str.replace(r'⅙', r'1/6', regex=True)
df['text'] = df['text'].str.replace(r'ﬀ', r'ff', regex=True)
df['text'] = df['text'].str.replace(r'ꝓ', r'ff', regex=True)
df['text'] = df['text'].str.replace(r'ů', r'u', regex=True)
df['text'] = df['text'].str.replace(r'̃', r'', regex=True)
df['text'] = df['text'].str.replace(r'̈', r'', regex=True)
df['text'] = df['text'].str.replace(r'ﬄ', r'ffl', regex=True)
df['text'] = df['text'].str.replace(r'⸗', r'-', regex=True)
df['text'] = df['text'].str.replace(r'ꝛ', r'r', regex=True)
for vok, uml in [('a', 'ä'), ('o', 'ö'), ('u', 'ü')]:
    df['text'] = df['text'].str.replace(vok+r'ͤ', uml, regex=True)

In [6]:
df["chunks"] = df["text"].fillna("").map(lambda t: split_for_byt5(t, max_len=512))
df_split = (df.explode("chunks", ignore_index=True).rename(columns={"chunks": "split_text"}))
df_split["text_byt_len"] = df_split["split_text"].map(byt5_byte_len)

assert df_split["text_byt_len"].max() <= 512

In [ ]:
df_split.to_csv(f"{tag}_split.csv", index=False)

In [8]:
res = parallel_generate_texts(df_split["split_text"], model_path="byt5-large-ocr-postcorrection", num_beams=4, max_length=576, batch_size=64*7)

Loading weights: 100%|██████████| 498/498 [00:00<00:00, 2377.49it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


progress: 91392/100000 (91.4%)

[W710 18:39:06.278308285 CUDACachingAllocator.cpp:508] expandable_segments: memory mapping failed with OOM on device 1 while trying to map 20971520 bytes (free: 8454144, total: 101976440832).


progress: 91840/100000 (91.8%)

[W710 18:39:13.521830551 CUDACachingAllocator.cpp:508] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 10551296, total: 101976440832).
[W710 18:39:14.120017822 CUDACachingAllocator.cpp:508] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 12648448, total: 101976440832).


progress: 92288/100000 (92.3%)

... (28 outputs left out)

[W710 18:51:17.756166423 CUDACachingAllocator.cpp:508] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 14745600, total: 101976440832).
[W710 18:51:32.820240864 CUDACachingAllocator.cpp:508] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 14745600, total: 101976440832).
[W710 18:51:32.609646264 CUDACachingAllocator.cpp:508] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 14745600, total: 101976440832).
[W710 18:51:34.520462077 CUDACachingAllocator.cpp:508] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 14745600, total: 101976440832).
[W710 18:51:36.222664288 CUDACachingAllocator.cpp:508] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 14745600, total: 101976440832).
[W710 18:51:38.162045899 CUDACachin

progress: 99280/100000 (99.3%)

[W710 18:53:09.561054651 CUDACachingAllocator.cpp:508] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 14745600, total: 101976440832).
[W710 18:53:10.765770265 CUDACachingAllocator.cpp:508] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 14745600, total: 101976440832).
[W710 18:53:12.394944426 CUDACachingAllocator.cpp:508] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 14745600, total: 101976440832).
[W710 18:53:14.467041795 CUDACachingAllocator.cpp:508] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 14745600, total: 101976440832).
[W710 18:53:16.102711811 CUDACachingAllocator.cpp:508] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 14745600, total: 101976440832).
[W710 18:53:19.762732599 CUDACachin

progress: 100000/100000 (100.0%)


In [4]:
df_split = pd.read_csv(f"{tag}_split_corrected.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'nwg_split_corrected.csv'

In [9]:
df_split["post_corrected"] = res['pred']
#df_temp["post_corrected"] = res['pred']

In [ ]:
per_region = (df_split.groupby(["metadata", "region_id"], sort=False)["post_corrected"].agg(" ".join).rename("post_corrected"))
df = df.merge(per_region, on=["metadata", "region_id"], how="left")

In [ ]:
df.drop(columns=["chunks"], inplace = True)

In [ ]:
df.to_csv(f"/home/lab6/nextcloud_fast/Jobads/final_selection/finished_csv/{tag}_postcorrected.csv", index=False)

In [166]:
df

,metadata,region_id,reading_order,text,min_x,max_x,min_y,max_y,post_corrected
0,nfp_18641015_008,region_0283,257,Jede ſolche Anzeige koſtet in-\nclufive Stempe...,2788,3269,128,859,Jede ſolche Anzeige koſtet in⸗\ncluſive Stempe...
1,nfp_18641015_008,region_0267,260,Ein im Schreib- und Rech-\nnungsfache ſowie de...,2824,3252,847,1140,Ein im Schreib⸗ und Rech⸗\nnungsfache ſowie de...
2,nfp_18641015_008,region_0212,263,Als COorrepetitor für Gym-\nnaſial-Gegenſtände...,2821,3244,1125,1243,Als Correpetitor für Gym⸗\nnaſial⸗Gegenſtände ...
3,nfp_18641015_008,region_0248,265,"Eine Lehrerin, mit Zeugnis-\nſen der angeſehen...",2818,3247,1228,1417,"Eine Lehrerin, mit Zeugnis⸗\nſen der angeſehen..."
4,nfp_18641015_008,region_0234,273,Ein Lehrer für Kauptschul-\nGegenſtände gegen ...,2802,3230,1872,2012,Ein Lehrer für Hauptschul⸗\nGegenſtände gegen ...
...,...,...,...,...,...,...,...,...,...
2013166,nfp_19390131_016,region_0204,250,Sebfürt. Engländerin\nKonverfanlon. Ueberſetzu...,1566,1814,3745,3831,Sepfürt. Engländerin\nKonverſation. Ueberſetzu...
2013167,nfp_19390131_016,region_0174,254,"ſucht Stelle. N. W., 17.\nGautergaſſe 8. 121—",1049,1300,3895,3950,"ſucht Stelle. N. W., 17.\nGautergaſſe 8. 121—5"
2013168,nfp_19390131_016,region_0254,255,Erfahrene Eehrerin\njucht Stelle für diei Nach...,1300,1557,3797,3965,Erfahrene Lehrerin\nſucht Stelle für die Nach⸗...
2013169,nfp_19390131_016,region_0231,262,Autterſproche) Pranz\nvnterr. diplem Lehrerin....,1808,2061,3078,3185,Autterſproche) Franz\ninterr. diplom. Lehrerin...


## Identify Job Ad Class

In [1]:
import torch
import gc
import threading
from transformers import ElectraForSequenceClassification, AutoTokenizer
import pickle
from tqdm.auto import tqdm
import pandas as pd

In [2]:
def run_inference(df):
    tokenizer = AutoTokenizer.from_pretrained(load_dir)
    n_gpus = torch.cuda.device_count()

    models = []
    for i in range(n_gpus):
        m = ElectraForSequenceClassification.from_pretrained(load_dir)
        m.to(f"cuda:{i}").eval()
        models.append(m)

    texts = df["post_corrected"].astype(str).tolist()
    predictions = [None] * len(texts)

    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    shards = [order[g::n_gpus] for g in range(n_gpus)]

    pbar = tqdm(total=len(texts), desc="Inference", unit="text")
    pbar_lock = threading.Lock()

    def run_shard(gpu_id, indices):
        model = models[gpu_id]
        device = f"cuda:{gpu_id}"
        for start in range(0, len(indices), BATCH_SIZE):
            batch_idx = indices[start:start + BATCH_SIZE]
            batch_texts = [texts[i] for i in batch_idx]
            inputs = tokenizer(batch_texts, return_tensors="pt", truncation=True, padding=True, max_length=512)
            inputs = {k: v.to(device, non_blocking=True) for k, v in inputs.items()}
            with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
                logits = model(**inputs).logits
            preds = logits.argmax(dim=1).cpu().tolist()
            for i, p in zip(batch_idx, preds):
                predictions[i] = p 

            with pbar_lock:
                pbar.update(len(batch_idx))

    threads = [threading.Thread(target=run_shard, args=(g, shards[g]))for g in range(n_gpus)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()

    pbar.close()
    return predictions

In [3]:
tag = 'nwj'

In [4]:
df = pd.read_csv(f"{tag}_postcorrected.csv")

In [5]:
load_dir = "electra_finetuned"
BATCH_SIZE = 256*3

In [6]:
predictions = run_inference(df)

gc.collect()
torch.cuda.empty_cache()

Inference:   0%|          | 0/492481 [00:00<?, ?text/s]

In [7]:
with open(f"{load_dir}/label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

/home/u2049/miniforge3/envs/jobads_env/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [8]:
df["prediction"] = predictions
df["prediction_label"] = label_encoder.inverse_transform(df["prediction"])

In [9]:
df.drop(columns=["prediction"], inplace = True)

In [10]:
df.to_csv(f"{tag}_categories.csv", index=False)

## Adjust Hypens in Text

In [1]:
import pandas as pd
import re
from pathlib import Path
import time
import listing_hyphen as lh

In [2]:
tag = 'wrz'
n_jobs = -1

In [3]:
#df = pd.read_csv(f"{tag}_categories.csv")
df = pd.read_csv(f"{tag}_extractions.csv")

In [234]:
df = lh.process_dataframe(df, n_jobs=n_jobs)

print(f"Rows changed by listing correction: {df['listing_changed'].sum():,}")
print(f"Rows changed by hyphen cleanup:     {df['hyphen_space_changed'].sum():,}")

[##############################] 100%  52,824/52,824 rows  0:02 elapsed  
Rows changed by listing correction: 1,669
Rows changed by hyphen cleanup:     2,188


In [235]:
df = df.drop(columns=["post_corrected_without_lb", "text_norm", "post_corrected_listings", "listing_changed", "hyphen_space_changed"])

In [5]:
df.to_csv(f"{tag}_extractions.csv", index=False)

Index(['Unnamed: 0', 'metadata', 'region_id', 'reading_order', 'text', 'min_x',
       'max_x', 'min_y', 'max_y', 'post_corrected', 'prediction_label',
       'salary', 'salary_importance', 'salary_period', 'unspecific_salary',
       'verpflegung', 'position', 'activity', 'attitude towards work',
       'background', 'interpersonal', 'job specific', 'language', 'gender',
       'iiif_link', 'post_corrected_final'],
      dtype='str')

## Create position mapping with orginial post corrected text

In [ ]:
import pandas as pd
from map_positions import add_pc_positions 

In [ ]:
tag = 'wrz'

In [ ]:
df = pd.read_csv(f"{tag}_extractions.csv", index_col=0, low_memory=False)
df["position_post_corrected"] = add_pc_positions(df)

In [ ]:
df.to_csv(f"{tag}_extractions.csv", index=False)

## Extract Job Information

In [6]:
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
import time
from queue import Empty
import ner_worker as nw
import pandas as pd
from IPython.display import display

In [237]:
tag = 'wrz'
PRINT_EVERY = 1
GPUS = [0, 1]
MAX_CHARS = 1500_000

In [238]:
#df = pd.read_csv(f"{tag}_categories.csv")
df = pd.read_csv(f"{tag}_extractions.csv")
#df = pd.read_excel(f"{tag}_no_lb_listing_hyphen_corrected.xlsx")
df = df.drop(columns=["Unnamed: 0"])
#df = df.drop(columns=["Unnamed: 0.1"])
#df = df.drop_duplicates(subset=['metadata', 'region_id', 'reading_order'])

In [239]:
df

,metadata,region_id,reading_order,text,min_x,max_x,min_y,max_y,post_corrected,prediction_label,...,position,activity,attitude towards work,background,interpersonal,job specific,language,gender,iiif_link,post_corrected_final
0,wrz_18500102_023,region_0013,19,"* annerbienen. el\nPPP\nVanfragen,bittet man z...",1988,2954,3209,3647,"PPP\nanfragen, bittet man zu 2 Perſonen.\nAnfr...",service_offer,...,"[(0, 3, 'PPP')]",NaN,NaN,NaN,NaN,NaN,NaN,"[(0, 3, 'neutral')]",https://iiif.onb.ac.at/images/ANNO/wrz18500102...,"PPP anfragen, bittet man zu 2 Personen. Anfrag..."
1,wrz_18500103_032,region_0139,30,"Eine zunge Dame, ganz verwaiſt und fremd in Wi...",230,1161,1818,2327,"Eine junge Dame, ganz verwaiſt und fremd in Wi...",job_search,...,"[(531, 541, 'Erzieherin')]",NaN,NaN,"[(5, 10, 'junge'), (17, 34, 'ganz verwaiſt und...",NaN,"[(62, 185, 'in allen weiblichen Kenntniſſen, i...",NaN,"[(531, 541, 'female')]",https://iiif.onb.ac.at/images/ANNO/wrz18500103...,"Eine junge Dame, ganz verwaist und fremd in Wi..."
2,wrz_18500103_032,region_0116,36,in allen beliebten Converſatienstänzen vortref...,227,1154,2371,2524,In allen beliebten Converſationstänzen vortref...,service_offer,...,NaN,NaN,NaN,NaN,NaN,"[(0, 63, 'In allen beliebten Converſationstänz...",NaN,NaN,https://iiif.onb.ac.at/images/ANNO/wrz18500103...,In allen beliebten Conversationstänzen vortref...
3,wrz_18500103_032,region_0121,42,"weiche ſehr geſchickt im Kleidermachen iſt, un...",223,1154,2576,2767,"Weiche ſehr geſchickt im Kleidermachen iſt, un...",job_search,...,"[(0, 6, 'Weiche'), (7, 11, 'ſehr'), (12, 21, '...",NaN,"[(0, 42, 'Weiche ſehr geſchickt im Kleidermach...",NaN,NaN,"[(48, 108, 'hier als\nKammerjungfer und Kleide...",NaN,"[(0, 6, 'neutral'), (7, 11, 'neutral'), (12, 2...",https://iiif.onb.ac.at/images/ANNO/wrz18500103...,"Weiche sehr geschickt im Kleidermachen ist, un..."
4,wrz_18500103_032,region_0134,50,von Manufacturwaaren können in einem hieſigen ...,216,1143,2878,3327,von Manufacturwaaren können in einem hieſigen ...,service_offer,...,NaN,"[(199, 260, 'Privat-Unterricht in der Engliſch...","[(322, 331, 'gründlich')]",NaN,NaN,NaN,"[(159, 193, 'Engliſche und Franzöſiſche Sprach...",NaN,https://iiif.onb.ac.at/images/ANNO/wrz18500103...,von Manufacturwaaren können in einem hiesigen ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52819,wrz_19501231_020,region_0377,170,Nn HERREN VON\nFORMAT mit guten Ver-\nbindunge...,200,606,3287,3679,Zu 2 HERREN VON\nFORMAT mit guten Ver-\nbindun...,job_offer,...,"[(5, 11, 'HERREN'), (12, 15, 'VON'), (16, 22, ...",NaN,NaN,NaN,NaN,"[(23, 83, 'mit guten Ver-\nbindungen zu Indust...",NaN,"[(5, 11, 'male'), (12, 15, 'neutral'), (16, 22...",https://iiif.onb.ac.at/images/ANNO/wrz19501231...,Zu 2 HERREN VON FORMAT mit guten Verbindungen ...
52820,wrz_19501231_020,region_0370,182,SUSSVARENINDU.\nSTRIE bzw. handel\n. a.! Chefb...,200,596,3808,4171,"AUSSTELLENINDUSTRIE,\nSTRIE bzw. Handel\na. a....",job_search,...,"[(0, 19, 'AUSSTELLENINDUSTRIE'), (45, 60, 'Che...","[(72, 113, 'Verrechnung.\nVerkauf desgleichen\...",NaN,NaN,NaN,"[(115, 142, 'beste Praxis\nund Kenntnisse')]",NaN,"[(0, 19, 'neutral'), (45, 60, 'neutral'), (61,...",https://iiif.onb.ac.at/images/ANNO/wrz19501231...,"AUSSTELLENINDUSTRIE, STRIE bzw. Handel a. a. C..."
52821,wrz_19501231_020,region_0382,265,"Wir suchen:\nFUr Salzburg dse, Luiman-\nnisch ...",1462,2193,3707,4066,"Wir suchen:\nFür Salzburg dsc, Luiman⸗\nnisch ...",vermittlung,...,"[(140, 149, 'Fachkraft')]",NaN,"[(125, 133, 'tüchtige')]","[(134, 139, 'junge')]",NaN,"[(38, 53, 'nisch gebildete'), (101, 110, 'lela...",NaN,"[(140, 149, 'neutral')]",https://iiif.onb.ac.at/images/ANNO/wrz19501231...,"Wir suchen: Für Salzburg dsc, Luimannisch gebi..."
52822,wrz_19501231_020,region_0343,269,An Dauerstellung Interessierlie richten hand-\...,1462,2189,4072,4218,An Dauerstellung Interessierte richten hand-\n...,job_offer,...,NaN,NaN,NaN,NaN,NaN,"[(39, 65, 'hand-\ngeschriebene Offerte')

In [240]:
#texts  = df["post_corrected"].astype(str).tolist()
texts  = df["post_corrected_final"].astype(str).tolist()
order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
shards = {g: order[k::len(GPUS)] for k, g in enumerate(GPUS)}
shard_texts = {g: [texts[i] for i in idx] for g, idx in shards.items()}

In [241]:
tasks = [(g, name) for g in GPUS for name in nw.MODELS]   # 2 GPUs x 3 models = 6 processes
ctx = mp.get_context("spawn")

with ctx.Manager() as manager:
    pq = manager.Queue()
    done = {t: 0 for t in tasks}
    total = {(g, name): len(shards[g]) for g, name in tasks}
    finished = set()
    start = last_print = time.time()
    # One output area that gets overwritten in place (works without ipywidgets).
    status = display({"text/plain": "starting workers..."}, raw=True, display_id=True)

    def print_progress():
        elapsed = time.time() - start
        n_done, n_total = sum(done.values()), sum(total.values())
        parts_str = "  ".join(
            f"G{g}:{name} " + ("done" if (g, name) in finished
                               else f"{100 * done[(g, name)] / max(total[(g, name)], 1):3.0f}%")
            for g, name in tasks)
        line = f"[{elapsed / 60:6.1f} min] {100 * n_done / n_total:5.1f}% overall | {parts_str}"
        status.update({"text/plain": line}, raw=True)

    # max_tasks_per_child=1 (Python 3.11+) guarantees one model per process; drop it on older Python.
    with ProcessPoolExecutor(max_workers=len(tasks), mp_context=ctx, max_tasks_per_child=1) as ex:
        futures = {
            ex.submit(nw.run_model, g, name, shard_texts[g], pq, MAX_CHARS): (g, name)
            for g, name in tasks
        }

        remaining = len(futures)
        while remaining:
            try:
                g, name, n = pq.get(timeout=0.5)
            except Empty:
                for f in futures:
                    if f.done() and f.exception():
                        raise f.exception()
                continue
            if n is None:
                remaining -= 1
                finished.add((g, name))
                print_progress()
                continue
            done[(g, name)] += n
            if time.time() - last_print >= PRINT_EVERY:
                print_progress()
                last_print = time.time()

        print_progress()

        # Put results back in the original text order.
        results = {name: [None] * len(texts) for name in nw.MODELS}
        labels, pipes = {}, {}
        for f, (g, name) in futures.items():
            ents_list, labs, pipe_names = f.result()
            labels[name], pipes[name] = labs, pipe_names
            for i, ents in zip(shards[g], ents_list):
                results[name][i] = ents

print("Active pipes per model:", pipes)

[   0.7 min] 100.0% overall | G0:salary done  G0:positions done  G0:requirements done  G1:salary done  G1:positions done  G1:requirements done

/home/u2049/miniforge3/envs/jobads_env/lib/python3.11/site-packages/spacy/util.py:971: UserWarning: [W095] Model 'de_pipeline' (0.0.0) was trained with spaCy v3.7.5 and may not be 100% compatible with the current version (3.8.14). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
/home/u2049/miniforge3/envs/jobads_env/lib/python3.11/site-packages/spacy/util.py:971: UserWarning: [W095] Model 'de_pipeline' (0.0.0) was trained with spaCy v3.7.5 and may not be 100% compatible with the current version (3.8.14). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
/home/u2049/miniforge3/envs/jobads_env/lib/python3.11/site-packages/sp

Active pipes per model: {'salary': ['transformer', 'ner'], 'positions': ['transformer', 'ner'], 'requirements': ['transformer', 'ner']}


In [242]:
for name, rows in results.items():
    for label in labels[name]:
        df[label] = [r.get(label) for r in rows]

In [243]:
df.to_csv(f"{tag}_extractions.csv", index=False)

## Extract Gender Information & Generate IIIF Link

In [13]:
import pandas as pd
from ast import literal_eval

In [244]:
tag = 'wrz'

In [245]:
df = pd.read_csv(f"{tag}_extractions.csv")

In [246]:
df['position'] = df['position'].astype(object)
mask = df['position'].notna()
df.loc[mask, 'position'] = df.loc[mask, 'position'].apply(literal_eval)

In [247]:
male_plurals = ["schmiede", "gesellen", "steinmetzen", "klempner-installateure", "köche", "küchenchefs", "elektro monteure", "betonbau-poliere", "chauffeure", "kupfer schmiede", "selchergesellen", "monteure", "ingenieure", "technikern", "konstrukteure", "architekten", "kunstschmiede", "elektromonteure", "asphalteure", "heizungsmoteure", "juristen", "polizisten", "betriebsingenieure"]

female_exceptions = ["stütze", "beamtenstochter", "kammerjungfer", "jungfer", "lehrerinstelle", "unterlehrerinenstelle", "lehrerinstellen", "sportmädl", "maschinnäherinen", "büglerinngen", "strickerin,", "vorrichterin nen", "mädches", "maschinenarbeiterinenn", "kammer jungfer", "lehrerinstelle", "nähterinen", "unterlehrerin⸗", "näherinen", "maschinen-näherinen", "kunststickerinen", "kellnerinen", "kammerjungser", "kindergärtnerin⸗", "kinderwärterinstelle", "uebungsschullehrerinstelle", "lehrerinstellen", "stall mägde", "herdmäd chen", "hausmägde", "mäd chen", "kinderfräul", "flickerin,", "stütze der hausfran", "erzieherinen", "bon nen", "herrschaftsköchin nen", "gemüseköchin,", "abspülerin nen", "biermäd chen", "mägde", "köchin für alles", "kindermäd chen", "stallmägde", "kellnerinposten", "hotelstubenmädch", "servierin nen", "dienstmädschen", "extramäd chen", "stubenmäd", "kellnerin nen", "bezirkshebammenstelle", "hausschneideren", "zimmermäd chen", "madchen", "bedienerln", "küchenangestellte", "hebamm", "stubenmadchen", "trastiderschrerßerln", "büglerinnnen", "stubenmäd- chen", "mächen für alles", "signorine", "institutrices", "kammerjungfern", "mandoline", "signorina", "stubenmäd chen", "française", "modistenlehrmüdchen", "küchenmäd chen", "abwascherin nen", "chefextramäd chei", "ammen", "weiber", "unterlehrerinen", "strickerinnnen", "kinderfräu", "buchbindermädcheu", "hefterin nen", "gauvernanten", "vorrichterrimnen", "kindsfranen", "kammerfungser", "spulerin nen", "modistenlehrmadchen", "lehrmadchen", "diplom-säuglingsschwester", "probirmamsel", "zeichnernnen", "kammerjnugfer", "köchiunen", "lehrerinnenstellen", "wäscherinn", "lehrerin nen"]
male_exceptions = ["damenfriseur", "damenschneider", "alphirte", "religionslehrers", "damenzuschneider", "oekonom", "miederzuschneider", "ober⸗thierarzt", "obergymnasiast", "magazineur,", "kontoristtt", "religlonslehrers", "schmied", "bezirksarztes", "gießermeisters", "wirt", "wagenschmied", "ferstwart", "wirth", "geräte wart", "stellvertreter des abteilungsleiters", "arzt", "privatdetektiv", "porzellanmaler -", "abonnentensammler", "stallpage", "musikdirectors", "spenglergeselle", "werkzeugschmied", "rechnungsfeldwebel", "compagnon", "rabbiners", "predigers", "wundarztes", "webergeselle", "gemeindearztes", "koch", "hotelwirt", "küchenwirt", "zeugschmied", "doktor der medizin", "sekundar⸗arztes", "gentleman", "pädagoge", "schmiedegeselle", "oberförsters", "försters", "reiterjägers", "volksschullehrers", "forstwirth", "claviervirtuose", "hammerschmied", "wagnergeselle", "privat⸗detectiv", "jungschmied", "landwirt", "postillon", "gutsschmied", "musiklehrers", "spezial⸗damenfriseur", "hufschmied", "zahnarzt", "lehrsunge", "dolmetscher-", "violindvirtuose", "geschäftsleiters", "geschäfts⸗leiter.", "landwirth", "verkäufer, versirt", "arztessohn", "virtuose", "diplom-landwirt", "pfarrhauswirt", "philologe", "feilschmied", "architekt", "gastwirt", "kupferschmied", "lehrfunge", "genosse", "männliche kräfte", "hilfsarztes", "stenograf", "kurschmied", "lehrjnunge", "hausfnecht", "hauskecht⸗", "wirt", "stricker,", "maͤnn", "lehrlunge"]

female_indicators = ['mädchen', 'frau', 'magd', 'fräulein', 'lady', 'bonne', 'dame', 'gouvernante', 'madam', 'mademoiselle', 'woman', 'mädel', 'mamsell', 'mamselle', 'mädl', 'schwester', 'mägde', 'tochter', 'mädch', 'zofe', 'signora', 'weib']
male_indicators = ['mann', 'herr', 'bursche', 'junge', 'knabe', 'knecht', 'bursch', 'geselle', 'signore', 'sohn', 'man', 'gentleman']

female_suffixes = ['in', 'innen', 'euse', 'mme', 'ice', 'esse', 'ière']
male_suffixes = ['er', 'eur', 'ren', 'ar', 'or', 'jung', 'är', 'ent', 'aph', 'ist', 'chef', 'et', 'at', 'ant', 'mis']

In [248]:
def get_gender_string(position_name):
    pos_lower = str(position_name).lower()
    if pos_lower in male_plurals:
        return 'male'
    elif pos_lower in female_exceptions:
        return 'female'
    elif pos_lower in male_exceptions:
        return 'male'
    elif any(indicator in pos_lower for indicator in female_indicators):
        return 'female'
    elif any(pos_lower.endswith(suffix) for suffix in female_suffixes):
        return 'female'
    elif any(pos_lower.endswith(suffix) for suffix in male_suffixes):
        return 'male'
    elif any(indicator in pos_lower for indicator in male_indicators):
        return 'male'
    else:
        return 'neutral'

def annotate_position_list(tuple_list):
    # Safety check: if the cell is empty/NaN, return it as-is
    if not isinstance(tuple_list, list):
        return tuple_list
        
    annotated_list = []
    # Unpack the 3 variables from your tuple
    for start, end, position_name in tuple_list:
        # Get the gender for just the text part
        gender = get_gender_string(position_name)
        
        # Append a new tuple containing the original digits + the text + the gender
        annotated_list.append((start, end, gender))
        
    return annotated_list

In [249]:
df['gender'] = df['position'].apply(annotate_position_list)

In [250]:
def generate_iiif(metadata, min_x, min_y, max_x, max_y):
    return f"https://iiif.onb.ac.at/images/ANNO/{metadata[:3]}{metadata[4:12]}/00000{metadata[-3:]}/{min_x},{min_y},{max_x - min_x},{max_y - min_y}/full/0/default.png"

In [251]:
df['iiif_link'] = df.apply(lambda row: generate_iiif(row['metadata'], row['min_x'], row['min_y'], row['max_x'], row['max_y']), axis=1)

In [252]:
df.to_csv(f"{tag}_extractions.csv")

### Remove Chracter Offsets

In [11]:
import ast

In [12]:
def parse_and_extract(val):
    if not isinstance(val, str):  
        return val
    lst = ast.literal_eval(val)
    if isinstance(lst, list):
        return [t[2] for t in lst]
    return lst

In [127]:
tag = 'dvb'

In [128]:
df = pd.read_csv(f"{tag}_extractions.csv")

/tmp/ipykernel_3670807/3596462181.py:1: DtypeWarning: Columns (0: salary_importance) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{tag}_extractions.csv")


In [129]:
df

,Unnamed: 0,metadata,region_id,reading_order,text,min_x,max_x,min_y,max_y,post_corrected,...,verpflegung,position,activity,attitude towards work,background,interpersonal,job specific,language,gender,iiif_link
0,0,dvb_18890102_012,region_0064,8,"FVomptoiriſt, ſelbſtſtändiger Arbeite!\nwünſch...",253,734,525,609,"Comptoiriſt, ſelbſtſtändiger Arbeiter\nwünſcht...",...,NaN,"[(29, 37, 'Arbeiter')]",NaN,NaN,NaN,NaN,"[(13, 37, 'ſelbſtſtändiger Arbeiter')]",NaN,"[(29, 37, 'male')]",https://iiif.onb.ac.at/images/ANNO/dvb18890102...
1,1,dvb_18890102_012,region_0113,9,Eine Hausadminiſtration inner-\nhalb Wien wird...,252,753,631,790,Eine Hausadminiſtration inner⸗\nhalb Wien wird...,...,NaN,"[(64, 71, 'beamten')]","[(5, 23, 'Hausadminiſtration')]",NaN,NaN,NaN,NaN,NaN,"[(64, 71, 'neutral')]",https://iiif.onb.ac.at/images/ANNO/dvb18890102...
2,2,dvb_18890102_012,region_0074,10,Ills Lehrling bei einem tüchtigen Me-\nA chani...,253,755,813,973,Als Lehrling bei einem tüchtigen Me-\nchaniker...,...,"[(50, 66, 'Wohnung und Koſt')]","[(4, 12, 'Lehrling')]",NaN,NaN,"[(77, 82, 'armer'), (91, 151, 'aus einer ſehr ...",NaN,NaN,NaN,"[(4, 12, 'neutral')]",https://iiif.onb.ac.at/images/ANNO/dvb18890102...
3,3,dvb_18890102_012,region_0112,11,Nuchhalterin. Ein Fräulein aus ſehr\ngütem Hau...,254,754,994,1154,Buchhalterin. Ein Fräulein aus ſehr\ngutem Hau...,...,NaN,"[(0, 12, 'Buchhalterin')]",NaN,NaN,"[(27, 47, 'aus ſehr\ngutem Hauſe')]",NaN,"[(49, 98, 'der einfachen und doppel⸗\nten Buch...",NaN,"[(0, 12, 'female')]",https://iiif.onb.ac.at/images/ANNO/dvb18890102...
4,4,dvb_18890102_012,region_0085,12,"Eine Verkäuferin oder Caſſiererin,\nwünſcht St...",253,750,1178,1286,"Eine Verkäuferin oder Caſſiererin,\nwünſcht St...",...,NaN,"[(5, 16, 'Verkäuferin'), (22, 33, 'Caſſiererin')]",NaN,NaN,NaN,NaN,NaN,NaN,"[(5, 16, 'female'), (22, 33, 'female')]",https://iiif.onb.ac.at/images/ANNO/dvb18890102...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111239,111239,dvb_19220827_016,region_0130,22,"AUNl Wirtſchafterin\njür ſrauenloſen, bẽſcheid...",252,743,2556,2938,"Eine Wirtſchafterin\nfür frauenloſen, beſcheid...",...,NaN,NaN,"[(20, 58, 'für frauenloſen, beſcheidenen Haush...",NaN,"[(72, 79, 'jüngere'), (99, 121, 'aus bürgerlic...","[(166, 174, 'einfache')]","[(250, 276, 'Angabe der Lohn⸗\nanſprüche'), (2...",NaN,NaN,https://iiif.onb.ac.at/images/ANNO/dvb19220827...
111240,111240,dvb_19220827_016,region_0108,24,"ſtrebſ., intell. Lehrlinge\nvon 15 bis 16 Jahr...",252,741,3110,3233,"Strebſ., intell. Lehrlinge\nvon 15 bis 16 Jahr...",...,NaN,"[(17, 26, 'Lehrlinge')]",NaN,NaN,"[(9, 16, 'intell.')]","[(0, 7, 'Strebſ.')]",NaN,NaN,"[(17, 26, 'neutral')]",https://iiif.onb.ac.at/images/ANNO/dvb19220827...
111241,111241,dvb_19220827_016,region_0122,88,Gesdäiisdiene--\nzum ſofortigen Cintritt. Ange...,2359,3321,1007,1280,Geschäftsdiener\nzum ſofortigen Eintritt. Ange...,...,NaN,"[(0, 15, 'Geschäftsdiener')]",NaN,NaN,NaN,NaN,NaN,NaN,"[(0, 15, 'male')]",https://iiif.onb.ac.at/images/ANNO/dvb19220827...
111242,111242,dvb_19220829_008,region_0154,150,Tüchtige olporteurs\nVapierbra\nſucht.\nVeuſtt...,2316,2817,2901,3285,Tüchtige Solporteurs\nPapierbranche\nſucht.\nD...,...,NaN,"[(9, 20, 'Solporteurs')]",NaN,"[(0, 8, 'Tüchtige')]",NaN,NaN,NaN,NaN,"[(9, 20, 'neutral')]",https://iiif.onb.ac.at/images/ANNO/dvb19220829...


In [130]:
df['salary'] = df['salary'].apply(parse_and_extract)
df['salary_importance'] = df['salary_importance'].apply(parse_and_extract)
df['salary_period'] = df['salary_period'].apply(parse_and_extract)
df['unspecific_salary'] = df['unspecific_salary'].apply(parse_and_extract)
df['verpflegung'] = df['verpflegung'].apply(parse_and_extract)
df['position'] = df['position'].apply(parse_and_extract)
df['activity'] = df['activity'].apply(parse_and_extract)
df['attitude towards work'] = df['attitude towards work'].apply(parse_and_extract)
df['background'] = df['background'].apply(parse_and_extract)
df['interpersonal'] = df['interpersonal'].apply(parse_and_extract)
df['job specific'] = df['job specific'].apply(parse_and_extract)
df['language'] = df['language'].apply(parse_and_extract)

NameError: name 'parse_and_extract' is not defined

In [16]:
df = df.dropna(subset=['position'])

In [17]:
df.explode("position").to_excel(f"{tag}_no_lb_listing_hyphen_corrected.xlsx")

In [2]:
df = pd.read_csv('HISCO_DHInfra_Discrepancy.csv')

In [3]:
df.to_excel(f"HISCO_DHInfra_Discrepancy.xlsx")

/tmp/ipykernel_626364/2199317075.py:1: UserWarning: Cell contents too long (37083), truncated to 32767 characters
  df.to_excel(f"HISCO_DHInfra_Discrepancy.xlsx")
/tmp/ipykernel_626364/2199317075.py:1: UserWarning: Cell contents too long (35686), truncated to 32767 characters
  df.to_excel(f"HISCO_DHInfra_Discrepancy.xlsx")
/tmp/ipykernel_626364/2199317075.py:1: UserWarning: Cell contents too long (34591), truncated to 32767 characters
  df.to_excel(f"HISCO_DHInfra_Discrepancy.xlsx")
/tmp/ipykernel_626364/2199317075.py:1: UserWarning: Cell contents too long (52432), truncated to 32767 characters
  df.to_excel(f"HISCO_DHInfra_Discrepancy.xlsx")
/tmp/ipykernel_626364/2199317075.py:1: UserWarning: Cell contents too long (35330), truncated to 32767 characters
  df.to_excel(f"HISCO_DHInfra_Discrepancy.xlsx")
